# Optional · Architecture visualization

This exploration is not part of the required run sequence. Return to [01_run_model_day.ipynb](../01_run_model_day.ipynb) for controlled benchmark outputs.


# 08 — Architecture visualization

Pass the same VisDrone image through each architecture. Print actual module names first, then attach hooks to selected real modules. This avoids assuming undocumented VMamba or framework internals.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

In [ ]:
if SMOKE_TEST:
    print("SMOKE_TEST: activation hooks require a trained model and are skipped.")
else:
    import torch, numpy as np, matplotlib.pyplot as plt
    from sklearn.decomposition import PCA
    def list_modules(model, contains=None):
        for name, module in model.named_modules():
            if not contains or contains.lower() in name.lower():
                print(name, module.__class__.__name__)


## Load a run and inspect names

For Faster R-CNN inspect backbone stages, neck/FPN, RPN, and RoI modules. For Swin inspect patch embedding and stage blocks. For VMamba search for `backbone`, `layers`, `blocks`, `ss2d`, or `op` only after seeing the installed names. For RT-DETR inspect backbone, encoder, decoder, query embeddings, and intermediate hidden states exposed by documented outputs.

In [ ]:
from src.training.checkpointing import RunRegistry
from src.models.registry import create_adapter
from src.utils.serialization import read_yaml
registry=RunRegistry(paths)
RUN_ID=None  # choose a completed run
if RUN_ID:
    run=next(r for r in registry.list_available_runs(status=None) if r["run_id"]==RUN_ID)
    run_dir=paths.run_dir(run["model_id"],RUN_ID); cfg=read_yaml(run_dir/"model_config.yaml")
    if run["framework"] in {"mmdetection","vmamba_mmdetection"}: cfg["resolved_framework_config"]=str(run_dir/"runtime_config.py")
    adapter=create_adapter(run["model_id"]); model=adapter.load_model(registry.load_checkpoint_from_registry(RUN_ID),cfg)
    list_modules(model)

## Effective receptive field

Backpropagate from a selected central activation to input pixels, normalize absolute gradients, and compare spread. Keep preprocessing and selected semantic level consistent across models.

## Same-image four-architecture comparison

This cell selects the best completed two-class run for each primary model, uses the same image, renders predictions side by side, and captures only real module names from the installed implementation.


In [ ]:
from src.data.dataloaders import CocoDetectionRecords
from src.evaluation.visualization import select_module_names, capture_module_outputs, plot_activation_views, draw_predictions
from IPython.display import display
primary_models = ["faster_rcnn_resnet50", "faster_rcnn_swin_t", "faster_rcnn_vmamba_t", "rtdetrv2_l"]
records = CocoDetectionRecords(paths.coco("2class")/"val", paths.coco("2class")/"annotations/instances_val.json")
shared_image = records[0]["image"]
keyword_map = {
    "faster_rcnn_resnet50": ["backbone.layer", "neck", "rpn_head", "roi_head"],
    "faster_rcnn_swin_t": ["patch_embed", "backbone.stages", "neck", "rpn_head", "roi_head"],
    "faster_rcnn_vmamba_t": ["patch_embed", "backbone.layers", "ss2d", "op", "neck"],
    "rtdetrv2_l": ["backbone", "encoder", "decoder", "query"],
}
for model_id in primary_models:
    candidates = registry.list_available_runs(model_id, "2class")
    if not candidates:
        print(model_id, "has no completed run")
        continue
    run = max(candidates, key=lambda item: float(item.get("best_validation_map", 0)))
    run_dir = paths.run_dir(model_id, run["run_id"])
    cfg = read_yaml(run_dir/"model_config.yaml")
    cfg["input_resolution"] = run["input_resolution"]
    if run["framework"] in {"mmdetection", "vmamba_mmdetection"}:
        cfg["resolved_framework_config"] = str(run_dir/"runtime_config.py")
    adapter = create_adapter(model_id)
    model = adapter.load_model(registry.load_checkpoint_from_registry(run["run_id"]), cfg)
    names = select_module_names(model, keyword_map[model_id], limit=16)
    outputs, handles = capture_module_outputs(model, names)
    prediction = adapter.predict([shared_image])[0]
    for handle in handles: handle.remove()
    print(model_id, "stage modules:", names)
    display(draw_predictions(shared_image, prediction, run["class_names"], threshold=0.25))
    try:
        display(plot_activation_views(outputs, maximum_modules=5))
    except RuntimeError as error:
        print(error)
